
# VariSetu — Model 1: Crowd Density Estimation
### Trained model (not pretrained) — CSRNet on UCF-QNRF

This notebook trains **one** of the three VariSetu ML components: the
crowd-density estimator that drives the Crowd Intelligence layer
(Fig.5.1, Layer 3) — live choke-point density, crush-risk alerting, and
the heatmap overlay on the command-centre dashboard.

**Architecture:** CSRNet (Li, Zhang & Chen, CVPR 2018) — a VGG16 frontend
(ImageNet-pretrained, first 10 conv layers) plus a dilated-convolution
backend. Fully convolutional, so it accepts any input resolution; output is
a single-channel density map whose sum is the estimated head-count. Chosen
over MCNN (weaker on very dense scenes) and over training a full Bayesian
Loss / DM-Count model from scratch (much heavier compute budget than a
hackathon Colab session affords) — CSRNet is the standard, well-documented
middle ground and matches the "trained model, not a toy, not a
research-frontier reimplementation" bar from the task brief.

**Dataset:** UCF-QNRF (Idrees et al., ECCV 2018) — 1,535 images, 1.25M
annotated head points, extreme density range (49–12,865 people/image),
which is the closest public proxy to Vari-scale crowds.

**What this notebook produces** (Section 9, "Export"):
- `crowd_density_model.pt` — trained weights
- `model_config.json` — everything the backend needs to reproduce inference exactly
- `verification_report.json` / `.csv` — MAE, MSE, RMSE, GAME(0–3), density-map
  PSNR/SSIM, and a breakdown by crowd-density tercile (low/medium/high) — not
  just a single headline accuracy number
- Sample density-map visualizations for a qualitative sanity check
- `Model1_CrowdDensity_output.zip` bundling all of the above for download

**Credit-efficiency note:** every heavy step (density-map ground-truth
generation, training) writes checkpoints/cache to Google Drive, not just
Colab's ephemeral disk. If your Colab session disconnects, re-run from the
top — Section 5 and Section 10 both skip work that's already cached/saved,
so a re-run is cheap, not a redo from zero.



## 1. Setup steps (read this before running)

1. **Get the dataset onto your Google Drive** (the notebook expects it there,
   not a fresh Kaggle download each session — Kaggle downloads are slow and
   you'd redo them every time the runtime resets):
   - Download the dataset once from Kaggle: `faihajalamtopu/ucf-qnrf`
     (https://www.kaggle.com/datasets/faihajalamtopu/ucf-qnrf)
   - Upload the extracted folder (or the zip) to your Google Drive, e.g. at
     `My Drive/VariSetu/UCF-QNRF/`
   - If you'd rather keep it as a zip on Drive, Section 4 below will unzip it
     into the Colab session automatically — just point `DRIVE_DATASET_PATH`
     at the `.zip` file instead of a folder.
2. **Runtime → Change runtime type → GPU** (T4 is enough; this notebook does
   not need A100-class compute).
3. Run cells **top to bottom, in order**. Do not skip Section 5
   (ground-truth generation) — the model cannot train without it.
4. Expected total runtime on a free-tier T4: ~25–40 min for ground-truth
   generation (one-time, cached after) + ~2–3 hrs for the full training
   schedule (Section 10 has a `QUICK_TEST` flag to sanity-check the whole
   pipeline in ~10 min on a tiny subset before committing to the full run).


In [ ]:

# ============================================================
# SECTION 2 — Install dependencies
# (torch/torchvision already present in Colab; the rest are not)
# ============================================================
!pip install -q opencv-python-headless h5py scikit-image tqdm
print("Dependencies installed.")


In [ ]:

# ============================================================
# SECTION 3 — Mount Google Drive & configure paths
# EDIT THESE TWO LINES for your own Drive layout, then leave the rest alone.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Either a folder (already extracted) or a .zip file on your Drive.
# Matches the layout you get from the Kaggle download (faihajalamtopu/ucf-qnrf):
#   UCF-QNRF/UCF-QNRF_ECCV18/{Test/, Train/, list, readme}
# Point this at whichever level you actually uploaded -- the outer "UCF-QNRF"
# folder, the inner "UCF-QNRF_ECCV18" folder, or a .zip of either one all
# work, because Section 4 below searches recursively for Train/ and Test/
# rather than assuming a fixed nesting depth. The "list" and "readme" files
# in that folder are not used by this notebook -- ignore them.
DRIVE_DATASET_PATH = "/content/drive/MyDrive/VariSetu/UCF-QNRF"  # <-- EDIT ME
# Where checkpoints, cached ground truth, and final outputs are saved so they
# survive a Colab disconnect. Created automatically if it doesn't exist.
DRIVE_WORKDIR = "/content/drive/MyDrive/VariSetu/Model1_CrowdDensity_workdir"  # <-- EDIT ME (or leave as-is)

import os
os.makedirs(DRIVE_WORKDIR, exist_ok=True)
print("Drive mounted. Workdir:", DRIVE_WORKDIR)


In [ ]:

# ============================================================
# SECTION 4 — Locate / extract the dataset
# Handles either a pre-extracted folder or a .zip on Drive. Verifies the
# expected UCF-QNRF layout (Train/ and Test/ subfolders, each with
# img_####.jpg + img_####_ann.mat pairs) before doing any real work, so a
# path typo fails fast in Section 4 instead of silently 20 minutes later.
# ============================================================
import glob
import zipfile

LOCAL_DATA_ROOT = "/content/ucf_qnrf_data"

if DRIVE_DATASET_PATH.endswith(".zip"):
    if not os.path.exists(LOCAL_DATA_ROOT):
        print("Extracting zip to local Colab disk (faster I/O than reading from Drive during training)...")
        os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)
        with zipfile.ZipFile(DRIVE_DATASET_PATH, 'r') as zf:
            zf.extractall(LOCAL_DATA_ROOT)
    search_root = LOCAL_DATA_ROOT
else:
    search_root = DRIVE_DATASET_PATH

def find_split_dir(root, split_name):
    '''UCF-QNRF ships as UCF-QNRF_ECCV18/{Train,Test}/... but Kaggle re-uploads
    vary the nesting -- search a few levels deep instead of hard-coding depth.'''
    candidates = glob.glob(os.path.join(root, "**", split_name), recursive=True)
    candidates = [c for c in candidates if os.path.isdir(c)]
    if not candidates:
        raise FileNotFoundError(
            f"Could not find a '{split_name}' folder under {root}. "
            "Check DRIVE_DATASET_PATH points at the UCF-QNRF root (or its zip)."
        )
    return candidates[0]

TRAIN_DIR = find_split_dir(search_root, "Train")
TEST_DIR = find_split_dir(search_root, "Test")

train_images = sorted(glob.glob(os.path.join(TRAIN_DIR, "*.jpg")))
test_images = sorted(glob.glob(os.path.join(TEST_DIR, "*.jpg")))

assert len(train_images) > 0, f"No .jpg files found in {TRAIN_DIR}"
assert len(test_images) > 0, f"No .jpg files found in {TEST_DIR}"

# Spot-check that every image has a matching annotation .mat file.
def ann_path_for(img_path):
    return img_path.replace(".jpg", "_ann.mat")

missing = [p for p in train_images[:20] if not os.path.exists(ann_path_for(p))]
assert not missing, f"Annotation .mat files missing for: {missing[:3]} ... check dataset layout."

print(f"Found {len(train_images)} train images, {len(test_images)} test images.")
print("Train dir:", TRAIN_DIR)
print("Test dir: ", TEST_DIR)
print("Sample files:", [os.path.basename(p) for p in train_images[:3]])


In [ ]:

# ============================================================
# SECTION 4b — Sanity-check one annotation file before committing to the
# full preprocessing run. UCF-QNRF's official ECCV18 release stores each
# image's head points as a MATLAB struct at mat['annPoints'] (shape Nx2,
# x,y columns). If Kaggle's re-upload used a different key name this will
# fail loudly here, in a few seconds -- not 20 minutes into Section 5.
# ============================================================
import scipy.io as sio

_sample_mat = sio.loadmat(ann_path_for(train_images[0]))
_keys = [k for k in _sample_mat.keys() if not k.startswith("__")]
print("Keys found in annotation .mat file:", _keys)

assert "annPoints" in _sample_mat, (
    f"Expected key 'annPoints' not found -- found {_keys} instead. "
    "Open one _ann.mat file with scipy.io.loadmat and check which key holds "
    "the Nx2 head-point array, then update `mat[\"annPoints\"]` in Section 5's "
    "load_annotation_points() to match before running the full preprocessing."
)
_sample_points = _sample_mat["annPoints"]
print(f"annPoints shape: {_sample_points.shape} (expect N x 2 -- x,y head coordinates)")
print(f"Sample image '{os.path.basename(train_images[0])}' has {_sample_points.shape[0]} annotated heads.")
print("Annotation format confirmed -- safe to proceed to Section 5.")


In [ ]:

# ============================================================
# SECTION 5 — Ground-truth density map generation (cached to Drive)
# ============================================================
# For each image: cap the longer side to MAX_DIM (multiple of 8, required
# because CSRNet downsamples by exactly 8x), scale the annotated head points
# to match, then build a geometry-adaptive Gaussian density map at that same
# resolution. "Geometry-adaptive" (per UCF-QNRF's own baseline, Idrees et al.
# 2018): each head's Gaussian sigma is derived from its distance to nearby
# heads (crowded regions -> smaller sigma / sharper peaks, sparse regions ->
# larger sigma), not one fixed sigma for the whole dataset -- important given
# UCF-QNRF's 49-to-12,865-people-per-image range.
#
# Implemented as a small local patch per point (not a full-image filter per
# point), so this stays fast even on images with thousands of heads.
# Cached as .npy (float16) to Drive so this ~25-40 min step never has to
# re-run after a disconnect.
# ============================================================
import numpy as np
import cv2
import scipy.io as sio
from scipy.spatial import cKDTree
from tqdm.auto import tqdm

MAX_DIM = 1024          # cap on longer side; also the model's effective input-size ceiling
K_NEIGHBORS = 3
BETA = 0.3
MIN_SIGMA, MAX_SIGMA = 2.0, 20.0

GT_CACHE_DIR = os.path.join(DRIVE_WORKDIR, "density_gt_cache")
RESIZED_IMG_CACHE_DIR = os.path.join(DRIVE_WORKDIR, "resized_images_cache")
os.makedirs(GT_CACHE_DIR, exist_ok=True)
os.makedirs(RESIZED_IMG_CACHE_DIR, exist_ok=True)


def round_down_to_8(v):
    return max(8, (v // 8) * 8)


def load_annotation_points(mat_path):
    mat = sio.loadmat(mat_path)
    # UCF-QNRF ECCV18 annotation format: key 'annPoints', shape (N, 2) as (x, y).
    points = mat["annPoints"].astype(np.float32)
    return points


def add_gaussian(density, x, y, sigma, h, w):
    radius = max(1, int(round(3 * sigma)))
    size = 2 * radius + 1
    k1d = cv2.getGaussianKernel(size, sigma)
    kernel = k1d @ k1d.T
    x0, x1 = x - radius, x + radius + 1
    y0, y1 = y - radius, y + radius + 1
    kx0, kx1, ky0, ky1 = 0, size, 0, size
    if x0 < 0:
        kx0 = -x0; x0 = 0
    if y0 < 0:
        ky0 = -y0; y0 = 0
    if x1 > w:
        kx1 = size - (x1 - w); x1 = w
    if y1 > h:
        ky1 = size - (y1 - h); y1 = h
    if x1 <= x0 or y1 <= y0:
        return
    density[y0:y1, x0:x1] += kernel[ky0:ky1, kx0:kx1]


def generate_density_map(points, h, w):
    density = np.zeros((h, w), dtype=np.float32)
    n = len(points)
    if n == 0:
        return density
    if n > 1:
        tree = cKDTree(points)
        k = min(K_NEIGHBORS + 1, n)
        dists, _ = tree.query(points, k=k)
        if dists.ndim == 1:
            dists = dists.reshape(-1, 1)
        avg_dist = dists[:, 1:].mean(axis=1) if k > 1 else np.full(n, MIN_SIGMA)
        sigmas = np.clip(BETA * avg_dist, MIN_SIGMA, MAX_SIGMA)
    else:
        sigmas = np.array([MIN_SIGMA])

    for (x, y), sigma in zip(points, sigmas):
        xi, yi = int(round(x)), int(round(y))
        if 0 <= xi < w and 0 <= yi < h:
            add_gaussian(density, xi, yi, float(sigma), h, w)
    return density


def preprocess_split(image_paths, split_name):
    print(f"Preprocessing split: {split_name} ({len(image_paths)} images)")
    manifest = []
    for img_path in tqdm(image_paths):
        stem = os.path.splitext(os.path.basename(img_path))[0]
        out_img_path = os.path.join(RESIZED_IMG_CACHE_DIR, split_name, f"{stem}.jpg")
        out_density_path = os.path.join(GT_CACHE_DIR, split_name, f"{stem}.npy")
        out_count_path = os.path.join(GT_CACHE_DIR, split_name, f"{stem}_count.txt")

        os.makedirs(os.path.dirname(out_img_path), exist_ok=True)
        os.makedirs(os.path.dirname(out_density_path), exist_ok=True)

        if os.path.exists(out_img_path) and os.path.exists(out_density_path) and os.path.exists(out_count_path):
            with open(out_count_path) as f:
                gt_count = float(f.read().strip())
            manifest.append({"image": out_img_path, "density": out_density_path, "count": gt_count})
            continue  # already cached from a previous run -- credit-efficient re-run

        img = cv2.imread(img_path)
        if img is None:
            print(f"  WARNING: could not read {img_path}, skipping.")
            continue
        h0, w0 = img.shape[:2]
        scale = min(1.0, MAX_DIM / max(h0, w0))
        new_w, new_h = round_down_to_8(int(w0 * scale)), round_down_to_8(int(h0 * scale))
        resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

        points = load_annotation_points(ann_path_for(img_path))
        gt_count = float(len(points))
        scaled_points = points * [new_w / w0, new_h / h0]

        density = generate_density_map(scaled_points, new_h, new_w)

        cv2.imwrite(out_img_path, resized)
        np.save(out_density_path, density.astype(np.float16))
        with open(out_count_path, "w") as f:
            f.write(str(gt_count))

        manifest.append({"image": out_img_path, "density": out_density_path, "count": gt_count})
    return manifest


train_manifest = preprocess_split(train_images, "Train")
test_manifest = preprocess_split(test_images, "Test")

print(f"Cached {len(train_manifest)} train + {len(test_manifest)} test ground-truth density maps.")
print(f"Cache location (survives disconnects): {GT_CACHE_DIR}")


In [ ]:

# ============================================================
# SECTION 6 — Train / validation split
# UCF-QNRF ships only Train/Test (no official val split). Hold out 10% of
# Train, stratified by count-tercile so validation isn't accidentally all
# sparse (or all extremely dense) scenes.
# ============================================================
import random

random.seed(42)

counts = np.array([m["count"] for m in train_manifest])
tercile_edges = np.percentile(counts, [33.3, 66.7])

def tercile_of(c):
    if c <= tercile_edges[0]:
        return 0
    elif c <= tercile_edges[1]:
        return 1
    return 2

by_tercile = {0: [], 1: [], 2: []}
for m in train_manifest:
    by_tercile[tercile_of(m["count"])].append(m)

val_manifest, fit_manifest = [], []
for t, items in by_tercile.items():
    random.shuffle(items)
    n_val = max(1, int(0.10 * len(items)))
    val_manifest.extend(items[:n_val])
    fit_manifest.extend(items[n_val:])

print(f"Training subset: {len(fit_manifest)} images")
print(f"Validation subset: {len(val_manifest)} images (stratified by density tercile)")
print(f"Held-out test set: {len(test_manifest)} images (untouched until Section 11)")


In [ ]:

# ============================================================
# SECTION 7 — Dataset class
# Training: random crop to a fixed patch (multiple of 8) + horizontal flip,
# so images of wildly different sizes can still be batched. The density-map
# crop is sum-pooled down by 8x to match CSRNet's output stride, using
# block-summing (NOT interpolation) so the local head-count is preserved
# exactly, not blurred away.
# Validation/Test: full image, batch size 1 (CSRNet is fully convolutional,
# so this is valid -- no cropping needed when not batching).
# ============================================================
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
CROP_SIZE = 384  # must be a multiple of 8

_normalize = T.Compose([T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])


def sum_pool_8x(density):
    '''Block-sum downsample by 8x -- preserves total count exactly (unlike cv2.resize).'''
    h, w = density.shape
    h8, w8 = h // 8, w // 8
    density = density[: h8 * 8, : w8 * 8]
    return density.reshape(h8, 8, w8, 8).sum(axis=(1, 3))


class CrowdDensityDataset(Dataset):
    def __init__(self, manifest, train=True):
        self.manifest = manifest
        self.train = train

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        item = self.manifest[idx]
        img = Image.open(item["image"]).convert("RGB")
        density = np.load(item["density"]).astype(np.float32)
        w, h = img.size

        if self.train:
            # Pad up if the cached (already-capped) image is smaller than the crop size.
            if h < CROP_SIZE or w < CROP_SIZE:
                pad_h, pad_w = max(0, CROP_SIZE - h), max(0, CROP_SIZE - w)
                img = T.functional.pad(img, (0, 0, pad_w, pad_h), fill=0)
                density = np.pad(density, ((0, pad_h), (0, pad_w)), mode="constant")
                w, h = img.size

            top = random.randint(0, h - CROP_SIZE)
            left = random.randint(0, w - CROP_SIZE)
            img = img.crop((left, top, left + CROP_SIZE, top + CROP_SIZE))
            density = density[top: top + CROP_SIZE, left: left + CROP_SIZE]

            if random.random() < 0.5:
                img = T.functional.hflip(img)
                density = np.ascontiguousarray(density[:, ::-1])

        img_t = _normalize(img)
        density_ds = sum_pool_8x(density)  # matches model's 1/8-resolution output
        density_t = torch.from_numpy(density_ds.copy()).unsqueeze(0).float()
        gt_count = float(item["count"])
        return img_t, density_t, gt_count


In [ ]:

# ============================================================
# SECTION 8 — Model: CSRNet
# Same architecture as backend/model_loader.py (CrowdDensity model_loader) --
# keep the two in sync if you ever change this cell.
# ============================================================
import torch.nn as nn
from torchvision.models import vgg16_bn


class CSRNet(nn.Module):
    def __init__(self, load_imagenet_weights=True):
        super().__init__()
        frontend_cfg = [64, 64, "M", 128, 128, "M", 256, 256, 256, "M", 512, 512, 512]
        backend_cfg = [512, 512, 512, 256, 128, 64]
        self.frontend = self._make_layers(frontend_cfg, in_channels=3)
        self.backend = self._make_layers(backend_cfg, in_channels=512, dilation=2)
        self.output_layer = nn.Conv2d(64, 1, kernel_size=1)
        if load_imagenet_weights:
            vgg = vgg16_bn(weights="IMAGENET1K_V1")
            self._load_vgg_frontend(vgg)

    @staticmethod
    def _make_layers(cfg, in_channels, dilation=1):
        layers = []
        for v in cfg:
            if v == "M":
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            else:
                layers += [nn.Conv2d(in_channels, v, kernel_size=3, padding=dilation, dilation=dilation),
                           nn.ReLU(inplace=True)]
                in_channels = v
        return nn.Sequential(*layers)

    def _load_vgg_frontend(self, vgg):
        vgg_layers = list(vgg.features.children())
        own_layers = list(self.frontend.children())
        bn_frontend_len = 33
        i = j = 0
        while i < len(own_layers) and j < bn_frontend_len:
            if isinstance(own_layers[i], nn.Conv2d) and isinstance(vgg_layers[j], nn.Conv2d):
                if own_layers[i].weight.shape == vgg_layers[j].weight.shape:
                    own_layers[i].weight.data = vgg_layers[j].weight.data.clone()
                    own_layers[i].bias.data = vgg_layers[j].bias.data.clone()
            i += 1
            j += 1

    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        x = self.output_layer(x)
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected -- go to Runtime > Change runtime type > GPU before training.")


In [ ]:

# ============================================================
# SECTION 9 — Metrics (beyond a single accuracy number, per the brief)
# ============================================================
from skimage.metrics import peak_signal_noise_ratio as sk_psnr
from skimage.metrics import structural_similarity as sk_ssim


def count_metrics(pred_counts, gt_counts):
    pred_counts, gt_counts = np.array(pred_counts), np.array(gt_counts)
    errors = pred_counts - gt_counts
    mae = np.mean(np.abs(errors))
    mse = np.mean(errors ** 2)
    rmse = np.sqrt(mse)
    # Mean absolute percentage error, guarding against the (rare) zero-count image.
    nonzero = gt_counts > 0
    mape = np.mean(np.abs(errors[nonzero]) / gt_counts[nonzero]) * 100 if nonzero.any() else float("nan")
    return {"MAE": float(mae), "MSE": float(mse), "RMSE": float(rmse), "MAPE_percent": float(mape)}


def game_metric(pred_density, gt_density, level):
    '''Grid Average Mean absolute Error (Guerrero-Gomez-Olmedo et al., 2015):
    split the density map into 4^level equal tiles, sum-count each tile
    independently, then average the per-tile absolute error. Rewards a model
    that gets the *spatial distribution* right, not just the whole-image
    total -- two very different heatmaps can have the same overall MAE.'''
    h, w = gt_density.shape
    n = 2 ** level
    tile_h, tile_w = h // n, w // n
    if tile_h == 0 or tile_w == 0:
        return abs(pred_density.sum() - gt_density.sum())
    total_error = 0.0
    for i in range(n):
        for j in range(n):
            gt_tile = gt_density[i * tile_h:(i + 1) * tile_h, j * tile_w:(j + 1) * tile_w]
            pred_tile = pred_density[i * tile_h:(i + 1) * tile_h, j * tile_w:(j + 1) * tile_w]
            total_error += abs(pred_tile.sum() - gt_tile.sum())
    return total_error / (n * n)


def density_map_quality(pred_density, gt_density):
    '''PSNR/SSIM between predicted and ground-truth density maps -- catches a
    model that gets the total count right by accident (e.g. one big blob in
    the wrong place) while MAE alone would call that a perfect prediction.'''
    data_range = max(gt_density.max(), pred_density.max(), 1e-6) - min(gt_density.min(), pred_density.min())
    if data_range <= 0:
        return {"PSNR": float("nan"), "SSIM": float("nan")}
    psnr = sk_psnr(gt_density, pred_density, data_range=data_range)
    ssim = sk_ssim(gt_density, pred_density, data_range=data_range)
    return {"PSNR": float(psnr), "SSIM": float(ssim)}


In [ ]:

# ============================================================
# SECTION 10 — Training loop
# QUICK_TEST=True runs one epoch on a small subset first -- use this to catch
# shape/config bugs in ~5-10 min before committing a Colab session to the
# full multi-hour run. Flip to False for the real training run.
# ============================================================
from torch.utils.data import DataLoader, Subset
import torch.optim as optim
import time
import json

QUICK_TEST = True   # <-- set to False once the quick test passes cleanly

EPOCHS = 3 if QUICK_TEST else 60
BATCH_SIZE = 8
LR = 1e-5
WEIGHT_DECAY = 5e-4
VAL_EVERY = 1
CHECKPOINT_PATH = os.path.join(DRIVE_WORKDIR, "checkpoint_last.pt")
BEST_MODEL_PATH = os.path.join(DRIVE_WORKDIR, "checkpoint_best.pt")
HISTORY_PATH = os.path.join(DRIVE_WORKDIR, "training_history.json")

fit_ds = CrowdDensityDataset(fit_manifest, train=True)
val_ds = CrowdDensityDataset(val_manifest, train=False)

if QUICK_TEST:
    fit_ds = Subset(fit_ds, list(range(min(64, len(fit_ds)))))
    val_ds = Subset(val_ds, list(range(min(16, len(val_ds)))))

fit_loader = DataLoader(fit_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

model = CSRNet(load_imagenet_weights=True).to(device)
criterion = nn.MSELoss(reduction="sum")
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

start_epoch = 0
best_val_mae = float("inf")
history = {"train_loss": [], "val_mae": [], "val_mse": []}

# Resume from a previous (possibly interrupted) run -- credit-efficient re-run.
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    start_epoch = ckpt["epoch"] + 1
    best_val_mae = ckpt["best_val_mae"]
    if os.path.exists(HISTORY_PATH):
        with open(HISTORY_PATH) as f:
            history = json.load(f)
    print(f"Resumed from checkpoint at epoch {start_epoch}, best_val_mae so far = {best_val_mae:.2f}")


@torch.no_grad()
def validate():
    model.eval()
    pred_counts, gt_counts = [], []
    for img, density, gt_count in val_loader:
        img = img.to(device)
        pred_density = model(img)
        pred_counts.append(pred_density.sum().item())
        gt_counts.append(gt_count.item())
    metrics = count_metrics(pred_counts, gt_counts)
    return metrics


for epoch in range(start_epoch, EPOCHS):
    model.train()
    epoch_start = time.time()
    running_loss = 0.0
    for img, density, gt_count in fit_loader:
        img, density = img.to(device), density.to(device)
        optimizer.zero_grad()
        pred = model(img)
        loss = criterion(pred, density) / img.size(0)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * img.size(0)

    train_loss = running_loss / len(fit_loader.dataset)
    history["train_loss"].append(train_loss)

    if (epoch + 1) % VAL_EVERY == 0:
        val_metrics = validate()
        history["val_mae"].append(val_metrics["MAE"])
        history["val_mse"].append(val_metrics["MSE"])
        scheduler.step(val_metrics["MAE"])

        improved = val_metrics["MAE"] < best_val_mae
        if improved:
            best_val_mae = val_metrics["MAE"]
            torch.save(model.state_dict(), BEST_MODEL_PATH)

        print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.3f} | "
              f"val_MAE={val_metrics['MAE']:.2f} | val_RMSE={val_metrics['RMSE']:.2f} | "
              f"time={time.time()-epoch_start:.0f}s | {'*BEST*' if improved else ''}")

    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "best_val_mae": best_val_mae,
    }, CHECKPOINT_PATH)
    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f)

print(f"Training loop finished. Best validation MAE: {best_val_mae:.2f}")
print("If QUICK_TEST was True and this ran cleanly, set QUICK_TEST = False and re-run this cell for the real training run.")



## Before continuing: turn off QUICK_TEST

If Section 10 just ran with `QUICK_TEST = True`, it only proved the pipeline
works end-to-end — it has **not** produced a usable model yet. Go back to
Section 10, set `QUICK_TEST = False`, and re-run that cell. Because
checkpoints are saved to Drive every epoch, you can let this run across
multiple Colab sessions (it resumes automatically) instead of needing one
uninterrupted multi-hour session.

Once the real training run has completed (or you've trained as many epochs
as your time budget allows), continue to Section 11.


In [ ]:

# ============================================================
# SECTION 11 — Full evaluation on the held-out Test set
# Loads the BEST checkpoint (by validation MAE, not just the last epoch --
# avoids reporting an overfit final-epoch number) and computes every metric
# from Section 9 across the entire UCF-QNRF Test split, plus a breakdown by
# density tercile so a strong overall MAE can't hide a model that only
# works on sparse scenes.
# ============================================================
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

test_ds = CrowdDensityDataset(test_manifest, train=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2)

test_counts = np.array([m["count"] for m in test_manifest])
test_tercile_edges = np.percentile(test_counts, [33.3, 66.7])

def test_tercile_of(c):
    if c <= test_tercile_edges[0]:
        return "low_density"
    elif c <= test_tercile_edges[1]:
        return "medium_density"
    return "high_density"

per_image_results = []
game_totals = {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0}
psnr_vals, ssim_vals = [], []

with torch.no_grad():
    for idx, (img, density, gt_count) in enumerate(tqdm(test_loader, desc="Evaluating on Test set")):
        img_dev = img.to(device)
        pred_density_t = model(img_dev)
        pred_density = pred_density_t.squeeze().cpu().numpy()
        gt_density_np = density.squeeze().numpy()

        pred_count = float(pred_density.sum())
        gt_c = float(gt_count.item())

        for L in game_totals:
            game_totals[L] += game_metric(pred_density, gt_density_np, L)

        quality = density_map_quality(pred_density, gt_density_np)
        if not np.isnan(quality["PSNR"]):
            psnr_vals.append(quality["PSNR"])
            ssim_vals.append(quality["SSIM"])

        per_image_results.append({
            "image": os.path.basename(test_manifest[idx]["image"]),
            "gt_count": gt_c,
            "pred_count": round(pred_count, 2),
            "abs_error": round(abs(pred_count - gt_c), 2),
            "density_tercile": test_tercile_of(gt_c),
        })

overall_pred = [r["pred_count"] for r in per_image_results]
overall_gt = [r["gt_count"] for r in per_image_results]
overall_metrics = count_metrics(overall_pred, overall_gt)

game_scores = {f"GAME_{L}": round(v / len(test_manifest), 3) for L, v in game_totals.items()}

by_tercile = {}
for tercile in ["low_density", "medium_density", "high_density"]:
    subset = [r for r in per_image_results if r["density_tercile"] == tercile]
    if subset:
        by_tercile[tercile] = count_metrics(
            [r["pred_count"] for r in subset], [r["gt_count"] for r in subset]
        )
        by_tercile[tercile]["n_images"] = len(subset)

verification_report = {
    "model": "CSRNet (VGG16-BN frontend + dilated backend)",
    "dataset": "UCF-QNRF Test split",
    "n_test_images": len(test_manifest),
    "overall_metrics": overall_metrics,
    "game_metrics": game_scores,
    "density_map_quality": {
        "mean_PSNR": float(np.mean(psnr_vals)) if psnr_vals else None,
        "mean_SSIM": float(np.mean(ssim_vals)) if ssim_vals else None,
    },
    "metrics_by_density_tercile": by_tercile,
}

print(json.dumps(verification_report, indent=2))


In [ ]:

# ============================================================
# SECTION 12 — Save metrics report, training curves, and sample visualizations
# ============================================================
import matplotlib.pyplot as plt
import csv

REPORT_DIR = os.path.join(DRIVE_WORKDIR, "verification_report")
os.makedirs(REPORT_DIR, exist_ok=True)

with open(os.path.join(REPORT_DIR, "verification_report.json"), "w") as f:
    json.dump(verification_report, f, indent=2)

with open(os.path.join(REPORT_DIR, "per_image_results.csv"), "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(per_image_results[0].keys()))
    writer.writeheader()
    writer.writerows(per_image_results)

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"])
axes[0].set_title("Training loss (sum-MSE per image)")
axes[0].set_xlabel("Epoch")
axes[1].plot(history["val_mae"], label="Val MAE")
axes[1].plot(history["val_mse"], label="Val MSE")
axes[1].set_title("Validation MAE / MSE")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "training_curves.png"), dpi=150)
plt.show()

# A handful of qualitative examples: predicted vs. ground-truth density maps,
# one from each density tercile, so the visual quality can be sanity-checked
# alongside the numeric metrics.
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
shown = {"low_density": False, "medium_density": False, "high_density": False}
row = 0
with torch.no_grad():
    for idx, (img, density, gt_count) in enumerate(test_loader):
        tercile = per_image_results[idx]["density_tercile"]
        if shown[tercile]:
            continue
        shown[tercile] = True

        pred_density = model(img.to(device)).squeeze().cpu().numpy()
        gt_density_np = density.squeeze().numpy()
        img_np = img.squeeze().permute(1, 2, 0).numpy()
        img_np = img_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        img_np = np.clip(img_np, 0, 1)

        axes[row, 0].imshow(img_np); axes[row, 0].set_title(f"{tercile}\nGT count: {per_image_results[idx]['gt_count']:.0f}")
        axes[row, 1].imshow(gt_density_np, cmap="jet"); axes[row, 1].set_title("Ground-truth density")
        axes[row, 2].imshow(pred_density, cmap="jet"); axes[row, 2].set_title(f"Predicted (count: {per_image_results[idx]['pred_count']:.0f})")
        for c in range(3):
            axes[row, c].axis("off")
        row += 1
        if row >= 3:
            break

plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "sample_predictions.png"), dpi=150)
plt.show()

print("Verification report, training curves, and sample visualizations saved to:", REPORT_DIR)


In [ ]:

# ============================================================
# SECTION 13 — Export artifacts for the backend (model + config + outputs zip)
# model_config.json follows the same shape/spirit as the Re-ID component's
# config file, so the backend developer sees one consistent pattern across
# both ML components.
# ============================================================
import shutil

EXPORT_DIR = os.path.join(DRIVE_WORKDIR, "export")
os.makedirs(EXPORT_DIR, exist_ok=True)

final_weights_path = os.path.join(EXPORT_DIR, "crowd_density_model.pt")
shutil.copy(BEST_MODEL_PATH, final_weights_path)

model_config = {
    "model_name": "CSRNet",
    "trained_on": "UCF-QNRF",
    "input_channels": 3,
    "output_stride": 8,
    "preprocessing": {
        "max_dimension": MAX_DIM,
        "normalize_mean": IMAGENET_MEAN,
        "normalize_std": IMAGENET_STD,
    },
    "density_alert_thresholds": {
        "moderate_count": 150,
        "critical_count": 400
    },
    "training": {
        "epochs_run": len(history["train_loss"]),
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "crop_size": CROP_SIZE,
        "best_val_mae": best_val_mae,
    },
    "test_set_metrics": verification_report["overall_metrics"],
    "test_set_game_metrics": verification_report["game_metrics"],
    "notes": [
        "density_alert_thresholds are starting defaults for the Pandharpur command-centre "
        "dashboard, tunable per choke-point -- NOT derived from training data (no ground-truth "
        "crush-threshold labels exist in UCF-QNRF). Recalibrate against real corridor footage "
        "before using these numbers for actual crush-risk alerting.",
        "Trained on UCF-QNRF, not Wari-corridor CCTV footage -- expect a fine-tuning pass on "
        "real corridor footage before production use (same caveat as the Re-ID component).",
    ],
}

with open(os.path.join(EXPORT_DIR, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=2)

# Bundle everything a reviewer or the backend team needs into one zip.
FINAL_ZIP_PATH = os.path.join(DRIVE_WORKDIR, "Model1_CrowdDensity_output.zip")
if os.path.exists(FINAL_ZIP_PATH):
    os.remove(FINAL_ZIP_PATH)

with zipfile.ZipFile(FINAL_ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(final_weights_path, "crowd_density_model.pt")
    zf.write(os.path.join(EXPORT_DIR, "model_config.json"), "model_config.json")
    zf.write(os.path.join(REPORT_DIR, "verification_report.json"), "verification_report/verification_report.json")
    zf.write(os.path.join(REPORT_DIR, "per_image_results.csv"), "verification_report/per_image_results.csv")
    zf.write(os.path.join(REPORT_DIR, "training_curves.png"), "verification_report/training_curves.png")
    zf.write(os.path.join(REPORT_DIR, "sample_predictions.png"), "verification_report/sample_predictions.png")

print(f"Done. Final deliverable zip saved to Drive: {FINAL_ZIP_PATH}")
print("Download it from Drive, or run the cell below to pull it directly into this Colab session's Files pane.")


In [ ]:

# ============================================================
# SECTION 14 (optional) — Direct download from Colab
# ============================================================
from google.colab import files
files.download(FINAL_ZIP_PATH)
